# Session 07: Practice Quiz

**Notebook 2 of 2: Self-check** · about 35 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded.

In [ ]:
from typing import Annotated, Literal

from fastapi import Depends, FastAPI, HTTPException, Path, Query, Request, status
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

print("ready")

## Q1 · Predict: where does each parameter come from?

```python
class NoteIn(BaseModel):
    text: str

@app.post("/teams/{team}/notes")
def add_note(team: str, note: NoteIn, urgent: bool = False, limit: int = 10) -> dict: ...
```

For each of `team`, `note`, `urgent` and `limit`: path, query string or body? Which are required?

In [ ]:
class NoteIn(BaseModel):
    text: str


app = FastAPI()


@app.post("/teams/{team}/notes")
def add_note(team: str, note: NoteIn, urgent: bool = False, limit: int = 10) -> dict:
    return {"team": team, "text": note.text, "urgent": urgent, "limit": limit}


operation = app.openapi()["paths"]["/teams/{team}/notes"]["post"]
for parameter in operation["parameters"]:
    print(f"{parameter['name']:<7} in {parameter['in']:<6} required={parameter['required']}")
print("body required:", operation["requestBody"]["required"])

client = TestClient(app)
print(client.post("/teams/network/notes?urgent=yes", json={"text": "VPN flapping"}).json())

<details>
<summary>Show answer</summary>

- `team` is in the path (`{team}`), so it comes from the **path**, and path parameters are always required.
- `note` is a Pydantic model, so it's the JSON **body**, required because it has no default.
- `urgent` and `limit` are simple types not in the path, so they're **query** parameters, optional because they have defaults.

`?urgent=yes` became `True`: query strings are converted in lax mode.

</details>

## Q2 · Predict: which status code?

Using the app below, predict the status code of each request **before** running the cell:

1. `GET /tickets/INC-1`
2. `GET /tickets/INC-404`
3. `GET /tickets/1`
4. `DELETE /tickets/INC-1`
5. `GET /ticket/INC-1`
6. `GET /tickets?limit=500`

In [ ]:
TICKETS = {"INC-1": {"id": "INC-1", "team": "network"}}

app = FastAPI()


@app.get("/tickets")
def list_tickets(limit: Annotated[int, Query(ge=1, le=100)] = 20) -> list[dict]:
    return list(TICKETS.values())[:limit]


@app.get("/tickets/{ticket_id}")
def get_ticket(ticket_id: Annotated[str, Path(pattern=r"^INC-\d+$")]) -> dict:
    if ticket_id not in TICKETS:
        raise HTTPException(status.HTTP_404_NOT_FOUND, f"ticket {ticket_id} not found")
    return TICKETS[ticket_id]


client = TestClient(app)
for method, url in [
    ("GET", "/tickets/INC-1"),
    ("GET", "/tickets/INC-404"),
    ("GET", "/tickets/1"),
    ("DELETE", "/tickets/INC-1"),
    ("GET", "/ticket/INC-1"),
    ("GET", "/tickets?limit=500"),
]:
    print(f"{method:<6} {url:<22} {client.request(method, url).status_code}")

<details>
<summary>Show answer</summary>

1. **200**: a known ticket.
2. **404**: the id is valid but unknown, and the route raises `HTTPException(404)`.
3. **422**: `1` fails the path pattern, so validation rejects it and the route never runs. "Malformed" (422) and "well-formed but missing" (404) are different answers.
4. **405 Method Not Allowed**: the path exists, but not for `DELETE`.
5. **404**: no route matches `/ticket/...` at all. FastAPI's own 404 body is `{"detail": "Not Found"}`.
6. **422**: `limit` breaks `le=100`.

</details>

## Q3 · Debug: the "successful" missing asset

The service desk reports that looking up an unknown asset returns `200 OK` with the body `null`, and their script crashes on it. Find the cause and fix it so that an unknown id returns `404` with `{"detail": "asset <id> not found"}`.

In [ ]:
class AssetOut(BaseModel):
    id: str
    team: str


ASSETS = {"db-03": AssetOut(id="db-03", team="database")}

app = FastAPI()


@app.get("/assets/{asset_id}")
def get_asset(asset_id: str) -> AssetOut | None:
    return ASSETS.get(asset_id)


client = TestClient(app)
response = client.get("/assets/printer-7")
print(response.status_code, response.text)

# Your fix: redefine get_asset (and the app) above, then check:
# assert response.status_code == 404 and response.json() == {"detail": "asset printer-7 not found"}

<details>
<summary>Show answer</summary>

`dict.get` returns `None` for a missing key, and the return type `AssetOut | None` **allows** `None`, so FastAPI serialised it as `null` with `200`. Nothing failed, so nothing told the client.

```python
@app.get("/assets/{asset_id}")
def get_asset(asset_id: str) -> AssetOut:
    if asset_id not in ASSETS:
        raise HTTPException(status_code=404, detail=f"asset {asset_id} not found")
    return ASSETS[asset_id]
```

Two lessons:

- `X | None` as a response type is rarely what you want for "get one thing". Absence is an error with its own status.
- The client's crash was a symptom; the API's contract was the bug. Status codes are part of the contract.

</details>

## Q4 · Debug: what leaked?

A security review flags this endpoint. What's wrong, and what's the smallest correct fix?

```python
class Integration(BaseModel):
    name: str
    url: str
    api_token: str

@app.get("/integrations/{name}")
def get_integration(name: str) -> Integration:
    return INTEGRATIONS[name]
```

<details>
<summary>Show answer</summary>

The response model is the **internal** model, so every field, including `api_token`, is serialised into the response, published in `/docs`, and probably logged by some proxy on the way.

Fix: a separate response model with only the public fields, used as the return type:

```python
class IntegrationOut(BaseModel):
    name: str
    url: str

@app.get("/integrations/{name}")
def get_integration(name: str) -> IntegrationOut:
    return INTEGRATIONS[name]        # filtered on the way out
```

Better still, store the token as `SecretStr` (Session 06), so even an accidental dump shows `'**********'`. And handle the missing name with a `404`, as in Q3.

</details>

## Q5 · Implement: an exception handler

The service layer raises `QuotaExceededError(team, limit)` when a team creates too many tickets in an hour. Without touching the route, make the API answer **`429 Too Many Requests`** with the body `{"detail": "team network exceeded 3 tickets per hour"}`.

In [ ]:
class QuotaExceededError(Exception):
    def __init__(self, team: str, limit: int) -> None:
        super().__init__(f"team {team} exceeded {limit} tickets per hour")
        self.team = team
        self.limit = limit


created: list[str] = []


def create_ticket_for(team: str) -> str:
    """Service code."""
    if created.count(team) >= 3:
        raise QuotaExceededError(team, 3)
    created.append(team)
    return f"INC-{len(created)}"


app = FastAPI()


@app.post("/teams/{team}/tickets", status_code=201)
def create(team: str) -> dict:
    return {"id": create_ticket_for(team)}


# Your handler and its registration here.


client = TestClient(app, raise_server_exceptions=False)
codes = [client.post("/teams/network/tickets").status_code for _ in range(4)]
print(codes)
last = client.post("/teams/network/tickets")
print(last.status_code, last.text)
# Expected: [201, 201, 201, 429], then 429 {"detail":"team network exceeded 3 tickets per hour"}

<details>
<summary>Show answer</summary>

```python
def quota_exceeded(request: Request, error: QuotaExceededError) -> JSONResponse:
    return JSONResponse(
        status_code=status.HTTP_429_TOO_MANY_REQUESTS,
        content={"detail": str(error)},
    )

app.add_exception_handler(QuotaExceededError, quota_exceeded)
```

Before the fix, the fourth request is a **500**: an unhandled exception. The handler turns a domain rule into an HTTP answer the client can act on (wait and retry), and the service stays free of HTTP. A real API would usually add a `Retry-After` header too: `JSONResponse(..., headers={"Retry-After": "3600"})`.

</details>

## Q6 · Predict: a leaking override

Two tests share one module-level `app`. What does the **second** test see, and why is that a problem?

In [ ]:
def get_region() -> str:
    return "eu"


app = FastAPI()


@app.get("/region")
def region(value: Annotated[str, Depends(get_region)]) -> dict:
    return {"region": value}


def test_us_region():
    app.dependency_overrides[get_region] = lambda: "us"
    assert TestClient(app).get("/region").json() == {"region": "us"}


def test_default_region():
    assert TestClient(app).get("/region").json() == {"region": "eu"}


for test in (test_us_region, test_default_region):
    try:
        test()
        print(f"{test.__name__}: passed")
    except AssertionError:
        print(f"{test.__name__}: FAILED")

<details>
<summary>Show answer</summary>

`test_default_region` **fails**: it gets `"us"`. `dependency_overrides` is a dict on the `app` object, and the first test never removed its entry, so the override leaks into every later test that uses the same app.

Worse, the outcome depends on test order: run `test_default_region` alone and it passes. That's the "passes alone, fails together" bug from Session 04's debugging lab.

Fix: set overrides in a fixture and clear them on teardown:

```python
@pytest.fixture
def us_client():
    app.dependency_overrides[get_region] = lambda: "us"
    yield TestClient(app)
    app.dependency_overrides.clear()
```

Or avoid shared state entirely: build a fresh app per test with an app factory, as the workbook's `create_app()` does.

</details>

## Q7 · Debug: `'State' object has no attribute 'service'`

This test errors. Explain why, and fix it with a one-word change to the test.

In [ ]:
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncIterator[None]:
    app.state.service = {"INC-1": "network"}
    yield


app = FastAPI(lifespan=lifespan)


@app.get("/tickets/{ticket_id}")
def get_ticket(ticket_id: str, request: Request) -> dict:
    return {"id": ticket_id, "team": request.app.state.service[ticket_id]}


client = TestClient(app)
try:
    print(client.get("/tickets/INC-1").json())
except AttributeError as error:
    print("AttributeError:", error)

<details>
<summary>Show answer</summary>

The lifespan runs only when the app **starts**, and a plain `TestClient(app)` doesn't start it. `app.state.service` was never set, so the route raised `AttributeError`, which `TestClient` re-raised.

Use the context-manager form:

```python
with TestClient(app) as client:
    print(client.get("/tickets/INC-1").json())
```

Entering the `with` block runs the startup code; leaving it runs the shutdown code. Any app with a lifespan needs this in tests.

</details>

## Q8 · Engineering judgment: where does the rule live, and which status?

A product rule: **"a ticket can't be closed until it has a resolution note."** A client sends `PATCH /tickets/INC-7` with `{"status": "closed"}` for a ticket with no note.

1. Where should the rule be implemented: the route function, the service, or the Pydantic request model?
2. Which status code should the client get: `400`, `409` or `422`?

<details>
<summary>Show answer</summary>

**1. The service.** The rule depends on the ticket's **current state** (does it have a note?), which the request model can't see. A request model can check what's *in the request*; it can't check it against stored data. Putting the rule in the route would tie it to HTTP, so a command-line tool or a nightly job could close tickets without notes. In the service, every front end gets it.

The service raises a domain exception, say `TicketNotResolvableError`, and an exception handler maps it.

**2. `409 Conflict`** is the most common choice: the request is valid on its own, but conflicts with the resource's current state. `422` is better kept for "this request is invalid by itself", which FastAPI already uses that way, so clients can treat all `422`s alike. `400` is a generic fallback that tells the client less.

Reasonable teams differ on 409 versus 422 here. What matters more is that the choice is **consistent** across the API and **documented** with `responses={409: ...}`.

</details>

## Q9 · Implement: a small endpoint

Add `GET /tickets/{ticket_id}/age` to the app below:

- `ticket_id` must match `^INC-\d+$` (otherwise `422`);
- an optional query parameter `unit`, either `"hours"` (the default) or `"days"`;
- the response is `{"id": ..., "age": <number>, "unit": ...}`, with days rounded to one decimal place;
- an unknown ticket gives `404`.

In [ ]:
AGES = {"INC-3007": 200, "INC-3001": 7}

app = FastAPI()

# Your endpoint here.

client = TestClient(app)
for url in [
    "/tickets/INC-3007/age",
    "/tickets/INC-3007/age?unit=days",
    "/tickets/INC-9/age",
    "/tickets/3007/age",
    "/tickets/INC-3001/age?unit=weeks",
]:
    response = client.get(url)
    print(url, response.status_code, response.json())
# Expected: 200 age 200 hours; 200 age 8.3 days; 404; 422; 422

<details>
<summary>Show answer</summary>

```python
class Age(BaseModel):
    id: str
    age: float
    unit: Literal["hours", "days"]

@app.get("/tickets/{ticket_id}/age")
def ticket_age(
    ticket_id: Annotated[str, Path(pattern=r"^INC-\d+$")],
    unit: Literal["hours", "days"] = "hours",
) -> Age:
    if ticket_id not in AGES:
        raise HTTPException(status_code=404, detail=f"ticket {ticket_id} not found")
    hours = AGES[ticket_id]
    age = hours if unit == "hours" else round(hours / 24, 1)
    return Age(id=ticket_id, age=age, unit=unit)
```

Notes:

- `Literal["hours", "days"]` as a query parameter gives validation **and** a drop-down in `/docs`.
- A small response model (`Age`) documents the response shape. Returning a bare `dict` works, but `/docs` then can't say what's in it.
- With `age: float`, `200` is returned as `200.0`. If that matters to clients, use `int | float`.

</details>

## Q10 · Compare: dependency injection

You know dependency injection from Spring (`@Autowired`, `@Service`) or ASP.NET Core (`services.AddScoped<...>()`). Answer briefly:

1. Where is FastAPI's "container", and how is a dependency registered?
2. Spring and ASP.NET Core have **singleton**, **scoped** (per request) and **transient** lifetimes. What lifetime does a plain `Depends(get_x)` have? How would you get a singleton?
3. How do tests replace a dependency in each framework?

<details>
<summary>Show answer</summary>

1. **There's no container and no registration.** A dependency is just a function named in a parameter's `Depends(...)`. FastAPI calls it for each request, after resolving *its* dependencies the same way. Wiring is explicit at the point of use, rather than by type lookup.
2. A plain dependency is **called on every request**, and its result is cached within that one request if several parameters use it, which is close to *scoped*. For a **singleton**, either cache the function (`@lru_cache` on `get_settings`) or create the object once at startup (in the lifespan, on `app.state`) and have the dependency return it, as the workbook's `get_service` does. A dependency written with `yield` gets setup and teardown around each request, like a scoped database session.
3. Spring: `@MockBean` or a test configuration. ASP.NET Core: `WebApplicationFactory` with `ConfigureTestServices`. FastAPI: `app.dependency_overrides[original] = replacement`, a plain dict, which must be cleared after the test (Q6).

The trade-off: FastAPI's approach is simple and visible, but the graph of what-depends-on-what lives in function signatures rather than in one configuration place.

</details>

---

**Done?** Go back over any question you got wrong; each points to a part of `01-concepts.ipynb`. Then continue **Project 02** with the workbook.